In [1]:
!pip install azure-eventhub

StatementMeta(, a38bddbc-301d-486c-a30a-80d5971d191a, 3, Finished, Available, Finished, False)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 317.1/317.1 kB 13.5 MB/s eta 0:00:00


In [ ]:
from azure.eventhub import EventHubProducerClient, EventData
from pyspark.sql.types import DecimalType, DoubleType
import pyspark.sql.functions as F
from datetime import datetime, date
from decimal import Decimal
import json

# Step 1: Load data
df_stream = spark.sql("SELECT * FROM FSI_IQ_Lakehouse.transactions LIMIT 1000")

# Step 2: Cast ALL DecimalType columns to DoubleType
for field in df_stream.schema.fields:
    if isinstance(field.dataType, DecimalType):
        df_stream = df_stream.withColumn(field.name, F.col(field.name).cast(DoubleType()))

# Step 3: Serializer
def json_serializer(obj):
    if isinstance(obj, (datetime, date)):
        return obj.isoformat()
    if isinstance(obj, Decimal):
        return float(obj)
    raise TypeError(f"Type {type(obj)} not serializable")

# Step 4: Connection
# Event Hub configuration
CONNECTION_STR = "#CONNECTION_STR_2#"
EVENT_HUB_NAME = "#EVENT_HUB_NAME_2#"

# Step 5: Send
rows = df_stream.collect()
print(f"Preparing to send {len(rows)} events...")

producer = EventHubProducerClient.from_connection_string(conn_str=CONNECTION_STR, eventhub_name=EVENT_HUB_NAME)

sent_count = 0

try:
    event_data_batch = producer.create_batch()
    for row in rows:
        event = EventData(json.dumps(row.asDict(), default=json_serializer))
        try:
            event_data_batch.add(event)
        except ValueError:
            producer.send_batch(event_data_batch)
            sent_count += len(event_data_batch)
            print(f"Batch sent. Total so far: {sent_count}")
            event_data_batch = producer.create_batch()
            event_data_batch.add(event)

    if len(event_data_batch) > 0:
        producer.send_batch(event_data_batch)
        sent_count += len(event_data_batch)

    print(f"✅ Successfully sent {sent_count} events to Event Hub")
finally:
    producer.close()

StatementMeta(, a38bddbc-301d-486c-a30a-80d5971d191a, 5, Finished, Available, Finished, False)

Preparing to send 103 events...
✅ Successfully sent 103 events to Event Hub
